# Introduction to interval arithmetic

In [ ]:
#    APM41012EP course notebook - Chapter 1 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Introduction to interval arithmetic
#    Author : M. Breden - (C) 2026

In [ ]:
from mpmath import mp, iv
import numpy as np
import plotly.graph_objects as go

Interval arithmetic is, as its name indicates, an arithmetic on intervals (of floating-point numbers) rather than on floating-point numbers.

## Constructing interval arithmetic

Usually, when performing computations on a computer, real numbers are approximated by floating-point numbers, that is, $x\in\mathbb{R}$ is replaced by $fl(x)$. In this exercise, the basic objects will not be plain floating-point numbers, but so-called *representable* intervals, i.e. intervals of the form $I = [x^-,x^+]$ where $x^-$ and $x^+$ are floating-point numbers. In this setting, a real number is replaced by a representable interval:

$$ I_x = \left[\nabla(x),\Delta(x)\right], $$

which, by definition of the roundings directed towards plus and minus infinity, contains the real number $x$.

In [ ]:
# An example showing how to define a representable interval approximating 0.1, using the iv.mpf command.
# Beware that, if x is not a floating-point number, iv.mpf(x) builds the interval representing fl(x). 
# In order to obtain the interval I_x defined above, one must use iv.mpf('x').

I0x = iv.mpf(0.125) # 0.125 = 2^{-4} is exactly representable in base 2, no problem
print("The interval I0x is", I0x)

I1x = iv.mpf(0.1)   # 0.1 is not exactly representable in base 2, it is fl(0.1) that is used here
print("The interval I1x is", I1x)

I2x = iv.mpf(1)/iv.mpf(10) # The right way to define I_{0.1} so that the interval does contain 0.1
print("The interval I2x is", I2x)

I3x = iv.mpf('0.1')        # A character string can also be used to get the same result
print("The interval I3x is", I3x)

In order to be able to compute with this new representation, the basic operations must first be rebuilt. Consider two representable intervals $I = [x^-,x^+]$ and $J=[y^-,y^+]$. The operations $\oplus$, $\ominus$, $\otimes$ and $\oslash$ between representable intervals are defined as follows:
- $I \oplus J$ is the smallest representable interval such that, $\forall x\in I$ and $\forall y\in J$, $x+y \in I \oplus J$,
- $I \ominus J$ is the smallest representable interval such that, $\forall x\in I$ and $\forall y\in J$, $x-y \in I \ominus J$,
- $I \otimes J$ is the smallest representable interval such that, $\forall x\in I$ and $\forall  y\in J$, $x\times y \in I \otimes J$, 
- $I \oslash J$ is the smallest representable interval such that, $\forall x\in I$ and $\forall y\in J$, $x\div y \in I \oslash J$.

Using the roundings directed towards plus and minus infinity ($\Delta$ and $\nabla$), the sets $I \oplus J$, $I \ominus J$, $I \otimes J$ and $I \oslash J$ can be determined in terms of $x^-$, $x^+$, $y^-$ and $y^+$:

- $I \oplus J = [\nabla(x^- + y^-) ,\Delta(x^+ + y^+)]$
- $I \ominus J = [\nabla(x^- - y^+) ,\Delta(x^+ - y^-)]$
- For multiplication and division, different cases must be distinguished. 

    If $x^-,y^->0$, then $I \otimes J = [\nabla(x^- \times y^-) ,\Delta(x^+ \times y^+)]$, 

    if $x^->0$ and $y^-\leq 0 \leq y^+$, then $I \otimes J = [\nabla(x^+ \times y^-) ,\Delta(x^+ \times y^+)]$, 
    
    if $x^->0$ and $y^- \leq y^+ \leq 0$, then $I \otimes J = [\nabla(x^+ \times y^-) ,\Delta(x^- \times y^+)]$, etc.
    
The results can be summarised as follows:
- $I \otimes J = [\nabla(\min(x^-y^-,x^-y^+,x^+y^-,x^+y^+)) ,\Delta(\max(x^-y^-,x^-y^+,x^+y^-,x^+y^+))]$
- $I \oslash J = I \otimes \left[\nabla\left(\frac{1}{y^+}\right),\Delta\left(\frac{1}{y^-}\right)\right]$, provided that $0\notin J$.


**Remark:** in order to be able to handle under/overflows and divisions by $0$, $\pm\infty$ is included in the list of values that the endpoints of a representable interval may take. Thus, if $x$ and $y$ are floating-point numbers such that $x+y$ is larger than the largest floating-point number, one defines $[0,x] \oplus [0,y] = [0,+\infty]$. The mpmath library makes it possible to perform such operations on representable sets.

In [ ]:
# Declaration of the intervals I and J using the iv.mpf command
I = iv.mpf([-1, 1])
J = iv.mpf([ 1, 3])

# A few examples of elementary operations on intervals
print(f"I   =", I)
print(f"J   =", J)
print(f"I+J =", I+J)
print(f"I-J =", I-J)
print(f"I*J =", I*J)
print(f"I/J =", I/J)

## Example: image of an interval under a function 

In this example, we compute the image of an interval under a function defined in three different ways.

Consider the three forms of the same function:
- $h(x) = x^2-x$
- $h(x) = x(x-1)$
- $h(x) = \left(x-\frac{1}{2}\right)^2 -\frac{1}{4}$

and the interval $I = [0,2]$.

In [ ]:
I = iv.mpf([0, 2])
print(f"I =", I)

def h1(x):
    return x**2-x
    
def h2(x):
    return x*(x-1)
    
def h3(x):
    return (x-1/2)**2 -1/4
    
print(f"1st interval containing h(I) with h defined by the 1st form : ", h1(I))
print(f"2nd interval containing h(I) with h defined by the 2nd form : ", h2(I))
print(f"3rd interval containing h(I) with h defined by the 3rd form : ", h3(I))

The image of $[0,2]$ under $h$ is $[-0.25,2]$. All the intervals obtained therefore do contain the image of $[0,2]$ under $h$, which is an essential property of interval arithmetic. However, in the first two cases the image is overestimated. This is due to the fact that $x$ appears several times in the corresponding formulas. In the first case for instance, since $x$ appears twice, one first computes $[0,2] \otimes [0,2] = [0,4]$, and must then subtract two intervals: $[0,4] \ominus [0,2] = [-2,4]$, without being able to use the fact that to each element of $[0,4]$ (an $x^2$) there corresponds a unique element of $[0,2]$ (an $x$), and that it would be enough to subtract these elements.

If no equivalent formula with a single occurrence of $x$ can be found, the initial interval can be subdivided in order to reduce the impact of these overestimations (see the cell below for an example with one subdivision, but this process can of course be repeated).

In [ ]:
I1 = iv.mpf([0, 1])
I2 = iv.mpf([1, 2])
print(f"I = I1 U I2, with I1 = {I1} and I2 = {I2}\n")

J1 = h1(I1)
J2 = h1(I2)
print(f"Using the first expression, h(I1) is contained in {J1} and h(I2) is contained in {J2}\n")

def union(I, J):
    ''' returns an interval containing the union of I and J '''
    return iv.mpf([min(I.a,J.a), max(I.b,J.b)])

J = union(J1, J2)

print(f"Taking the union of these two intervals, we conclude that h(I) is contained in {J}, which is a sharper enclosure than the interval {h1(I)} obtained without subdivision.")

More generally, given a function $f:\mathbb{R}\to\mathbb{R}$ and a representable interval $I = [x^-,x^+]$, we want to be able to compute a representable interval, denoted by $f(I)$, which contains $\{f(x),\ x\in I\}$. Note that such an interval is not unique; in practice one tries to make it as small as possible.

## Application to controlling the error of floating-point computations



Truncation errors are made at every floating-point operation. They are usually negligible. Interval arithmetic makes it possible to obtain bounds on these errors. Instead of replacing $x\in\mathbb{R}$ by its floating-point approximation $fl(x)$, it is replaced by the representable interval $I_x = \left[\nabla(x),\Delta(x)\right]$ which, by definition of the roundings directed towards plus and minus infinity, contains the real number $x$. Then, operations are carried out on the interval $I_x$ rather than on $fl(x)$, and the properties of interval arithmetic allow us to assert that the final interval contains the quantity we are looking for!

In [ ]:
# Floating-point computation: initialise a 
a = 0.1

# test whether 3*a = 0.3 (here "a==b" returns the boolean True if a=b and False otherwise)
print(f"The equality 3*a=0.3 is ", 3*a==0.3)
print(f"Because of rounding errors, we do not have exactly 3a=0.3.\n")

# Computation with intervals: initialise Ia 
Ia = iv.mpf('0.1')
print(f"The interval 3*Ia = ", 3*Ia)
print(f"When 0.1 is replaced by the associated representable interval Ia and 3*Ia is computed, we do obtain an interval containing 0.3.")

## Obtaining certified results

Consider the two forms of the same function:
$$ f(x,y) = 9 x^4  - y^4 + 2 y^2 \quad \text{and} \quad f(x,y) = (3 x^2 - y^2)(3 x^2 + y^2) + 2 y^2 $$

Evaluating the function at $x=40545.$ and $y=70226.$ gives the following results:

In [ ]:
def f1(x,y):
    return 9*x**4-y**4+2*y**2

def f2(x,y):
    return (3*x**2-y**2)*(3*x**2+y**2)+2*y**2

x = 40545.
y = 70226.

print(f1(x,y))
print(f2(x,y))

If intervals are used instead:

In [ ]:
Ix = iv.mpf(x)
Iy = iv.mpf(y)

print(f1(Ix,Iy))
print(f2(Ix,Iy))

Interval arithmetic alone does not *solve* the problem of rounding errors, but it yields certified results. With the first formulation, in float64 we obtained $f(x,y)=1160$, with no indication that this result could potentially be wrong. The same formulation with interval arithmetic does not give precise information either, but at least we are certain that the correct value is contained in the interval $[-7032,5256]$. On the other hand, interval arithmetic makes it possible to guarantee the absence (or the small influence) of rounding errors when these are indeed negligible, as is the case with the second formulation, where we have a proof that the correct value is $1$.

These advantages of course come with drawbacks: on the one hand computations with interval arithmetic are more expensive, and on the other hand they may lead to significant overestimations of the errors.

## A simple example of a computer-assisted proof using interval arithmetic

Consider the function $g:\mathbb{R}\to\mathbb{R}$ defined by 

$$ g(x) = \exp(\sin(20x)) + x^4 +\frac{x}{1+x^2} -\frac{4}{100}. $$

We seek to prove the following statement: $g(x)>0$ for all $x\in\mathbb{R}$.

In [ ]:
def g(x):
    """ Definition of the function g """
    return np.exp(np.sin(20*x)) + x**4 + x/(1.+x**2) - 4./100.

On the interval $[-2,2]$, the function $g$ can be plotted:

In [ ]:
# creation of the data for the graphical output
x   = np.linspace(-2,2,1000)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=g(x), name='g(x)'))
fig.show()

A floating-point number $M>1$ such that $g(x)>0$ for all $\vert x\vert>M$ can be determined explicitly.

Indeed, we have $\exp(\sin(20x)) \geq \exp(-1)$, and a quick study of the function shows that $\displaystyle \frac{x}{1+x^2} \geq -\frac{1}{2}$. We deduce that

$$g(x) \geq \exp(-1) + x^4 -\frac{1}{2} - \frac{4}{100} \geq x^4 -1,$$

and therefore that g(x)>0 for all $\vert x\vert >1$; in other words we may take $M=1$.

An enclosure of $g$ on $[-M,M]$ can then be obtained using interval arithmetic:

In [ ]:
def ig(x):
    """ Definition of the function g using the basic functions on intervals """
    return iv.exp(iv.sin(20*x)) + x**4 + x/(1.+x**2) - 4./100.

M = 1
I = iv.mpf([-M,M])
print(ig(I))

The interval obtained is guaranteed to contain $g([-1,1])$, but it contains negative numbers, so we cannot conclude yet. 

We can split $[-M,M]$ into several subintervals, and obtain an enclosure of $g$ on each of them using interval arithmetic. By repeating this procedure, we obtain a proof that $g$ remains strictly positive on $[-M,M]$.

In [ ]:
def test_positivity(I,tol):
    print(f"Testing the positivity of g on the interval I = {I}")
    if ig(I).a>0:
        print(f"I = {I}, g(I) contained in {ig(I)}")
        return True 
    else: #If we cannot conclude yet that g(I)>0, we split I into two subintervals
        if I.delta>tol: 
            I1=iv.mpf([I.a,I.mid])
            I2=iv.mpf([I.mid,I.b])
            return min(test_positivity(I1,tol),test_positivity(I2,tol)) #We split in two and look at each piece
        else: #If the input interval has length smaller than tol, we stop (to make sure the algorithm does not loop forever)
            print(f"I = {I}, g(I) contained in {ig(I)}")
            return False #this False does not mean that the function necessarily takes negative values, only that we did not manage to prove positivity
        
tol = 1e-2
test_positivity(I,tol)

## References 

[1] IEEE standard for floating-point arithmetic. IEEE Std 754-2008, pages 1–70, Aug 2008.

[2] W. Tucker. Validated numerics. Princeton University Press, Princeton, NJ, 2011. A short introduction to rigorous computations.

[3] R. Moore. Interval analysis (Vol. 4). Englewood Cliffs: Prentice-Hall, 1966.